In [19]:
import numpy as np

In [20]:
# Modificación en el script de pruebas:
def f_2(X, Y):  # <--- Ahora acepta los 2 argumentos separados que le envía el *p
    r2 = X**2 + Y**2
    z = (r2)**0.25 * (np.sin(50 * (r2)**0.1)**2 + 1)
    return z



In [21]:
class PSO():

    def __init__(self, c1=2, c2=2, w= 0.7, strategy = "global"):
        self.c1 = c1
        self.c2 = c2
        self.w = w
        self.strategy = strategy

        if self.strategy == "global":
            self.update_positions = self._global_update
            self.update_group_stats= self._stats_global_update
        
        elif self.strategy == "local":
            self.update_positions = self._local_update
            self.update_group_stats = self._stats_local_update

    def run (self, n_particles, function, bounds, max_iters= 1000, neigh_size = None):

        self.n_particles = n_particles
        self.dim = len(bounds)

        self._validate_inputs(n_particles, neigh_size)
        
        self.function = function
        self.bounds = bounds

        
        positions, velocities = self._initialize_population()
        fitness = np.array([self.function(*p) for p in positions]) #calculamos el fitnes de cada una

        #memoria personal
        pbest_pos = positions.copy()
        pbest_fit = fitness.copy()

        #memoria grupal (g de grupal no de global)
        gbest_pos = np.zeros_like(positions)
        gbest_fit = np.zeros_like(fitness)

        self.update_group_stats(pbest_pos, pbest_fit, gbest_pos, gbest_fit) 
        

        for _ in range(max_iters):
            for i in range(self.n_particles):
                self.update_positions(positions, velocities, i, pbest_pos, gbest_pos)
            fitness = np.array([self.function(*p) for p in positions]) #recalculamos el fitnes de cada una

            self._update_personal_stats(positions, fitness, pbest_pos, pbest_fit)
            self.update_group_stats(pbest_pos, pbest_fit, gbest_pos, gbest_fit)

        best_idx = np.argmin(pbest_fit) #considerando minimzar
        return pbest_pos[best_idx], pbest_fit[best_idx]



    
    def _global_update(self, positions, velocities, idx_particle, personal_best, global_best_pos):
        r1 = np.random.random(self.dim) #vector de 0,1
        r2 = np.random.random(self.dim) #idem

        velocities[idx_particle, :] = (self.w * velocities[idx_particle,:]+
                                       self.c1 * r1 *(personal_best[idx_particle] - positions[idx_particle])+
                                       self.c2 * r2 * (global_best_pos[idx_particle] - positions[idx_particle]))
        
        positions[idx_particle, :] += velocities[idx_particle, :] #actualizamos posicion
        positions[idx_particle, :] = np.clip(positions[idx_particle, :], self.bounds[:, 0], self.bounds[:, 1]) #q no se vaya


    
    def _initialize_population(self):
        # matriz:
        # - fila: particula
        # - columna: posicion en la dimension
        position = np.random.uniform(
            low=self.bounds[:, 0],  
            high=self.bounds[:, 1],
            size=(self.n_particles, self.dim)
        )
        velocities = np.zeros((self.n_particles, self.dim))

        return position, velocities

    def _validate_inputs(self, n_particles, neigh_size):
        if self.strategy == "global":
            self.neigh_size = n_particles
        elif self.strategy == "local":
            if neigh_size is not None and neigh_size < n_particles:
                self.neigh_size = neigh_size
            else:
                raise ValueError("invalid neighborhood size for locla strategy")
        else:
            raise ValueError("invalid strategy")

    def _local_update(self, positions, velocities, idx_particle, personal_best, global_best_pos):
        # Como estructuramos los líderes por fila, la matemática de actualización es idéntica al global
        r1 = np.random.random(self.dim)
        r2 = np.random.random(self.dim)

        velocities[idx_particle, :] = (self.w * velocities[idx_particle,:]+
                                       self.c1 * r1 *(personal_best[idx_particle] - positions[idx_particle])+
                                       self.c2 * r2 * (global_best_pos[idx_particle] - positions[idx_particle]))
        
        positions[idx_particle, :] += velocities[idx_particle, :]
        positions[idx_particle, :] = np.clip(positions[idx_particle, :], self.bounds[:, 0], self.bounds[:, 1])

    def _stats_global_update(self, pbest_pos, pbest_fit, gbest_pos, gbest_fit):
        gbest_idx = np.argmin(pbest_fit)
        gbest_pos[:] = pbest_pos[gbest_idx] #es una matriz con el numero repetido
        gbest_fit[:] = pbest_fit[gbest_idx] #idem

        

    def _stats_local_update(self, pbest_pos, pbest_fit, gbest_pos, gbest_fit):
        # Implementación de topología de anillo simétrica basada en el tamaño de vecindad k (self.neigh_size)
        half_k = self.neigh_size // 2

        for i in range(self.n_particles):
            # Calculamos los índices de los vecinos usando el operador módulo (%) para simular el anillo cerrado
            neighbors_idx = [(i + j) % self.n_particles for j in range(-half_k, half_k + 1)]
            
            # Buscamos cuál de esos vecinos específicos tiene el mejor rendimiento histórico personal
            best_neighbor_idx = neighbors_idx[np.argmin(pbest_fit[neighbors_idx])]
            
            # Guardamos el mejor de SU vecindad en la fila correspondiente a la partícula i
            gbest_pos[i, :] = pbest_pos[best_neighbor_idx]
            gbest_fit[i] = pbest_fit[best_neighbor_idx]
            

    def _update_personal_stats(self, positions, fitness, pbest_pos, pbest_fit):
        #actualizar mejor personal
        for i in range(self.n_particles):
            if fitness[i] < pbest_fit[i]:
                pbest_fit[i] = fitness[i]
                pbest_pos[i, :] = positions[i, :].copy()

In [22]:
pso_global = PSO(strategy="global")

# 3. Definimos los límites (bounds) para X (dim 0) e Y (dim 1)
# Probamos en el rango de [-10, 10] para ambas variables
bounds = np.array([
    [-10.0, 10.0],  # Límites para X
    [-10.0, 10.0]   # Límites para Y
])

# 4. Configuramos los hiperparámetros de la ejecución
n_particles = 40
max_iters = 250

print("=> Ejecutando PSO Global...")
best_position, best_fitness = pso_global.run(
    n_particles=n_particles,
    function=f_2,
    bounds=bounds,
    max_iters=max_iters
)

# 5. Mostramos los resultados obtenidos
print("\n" + "="*40)
print("¡EJECUCIÓN TERMINADA CON ÉXITO!")
print("="*40)
print(f"Mejor posición encontrada (X, Y): {best_position}")
print(f"Valor de fitness en esa posición: {best_fitness:.6f}")
print("-"*40)
print("Nota: El óptimo teórico global de f_2 está en (0, 0) con fitness = 0.0")
print("="*40)

=> Ejecutando PSO Global...

¡EJECUCIÓN TERMINADA CON ÉXITO!
Mejor posición encontrada (X, Y): [7.86749122e-10 7.68094079e-09]
Valor de fitness en esa posición: 0.000164
----------------------------------------
Nota: El óptimo teórico global de f_2 está en (0, 0) con fitness = 0.0


In [24]:
# Instanciamos en modo local con una vecindad de 3 partículas (izquierda, ella misma, derecha)
pso_local = PSO(strategy="local")

best_position, best_fitness = pso_local.run(
    n_particles=40,
    function=f_2,
    bounds=bounds,
    max_iters=250,
    neigh_size=3  # <-- Importante definirlo en la estrategia local
)

print("\n" + "="*40)
print("¡EJECUCIÓN TERMINADA CON ÉXITO!")
print("="*40)
print(f"Mejor posición encontrada (X, Y): {best_position}")
print(f"Valor de fitness en esa posición: {best_fitness:.6f}")
print("-"*40)
print("Nota: El óptimo teórico global de f_2 está en (0, 0) con fitness = 0.0")
print("="*40)


¡EJECUCIÓN TERMINADA CON ÉXITO!
Mejor posición encontrada (X, Y): [-2.08929876e-07  7.51714740e-07]
Valor de fitness en esa posición: 0.000900
----------------------------------------
Nota: El óptimo teórico global de f_2 está en (0, 0) con fitness = 0.0
